# 05b — Alert Feature Tables (dual grain: route + stop)

Standalone, **additive-only** build from the `service_alerts` GTFS-RT archive.
Does not read from or write to notebooks 05/06/07, the `ml_features`
snapshot, or the weather table (`05c`).

## Why dual grain, not a single resolved table

The `ml_features` fact table is trip x stop x time and already carries
**both** `route_id` and `stop_id`. A prior diagnostic found that
`informed_entity` resolves cleanly in two disjoint ways:

- **44.9%** of alert occurrences carry >=1 `route_id` directly.
- **54.3%** carry a `stop_id` but **no** `route_id` at all.
- **~0.8%** carry no entity of any kind (dropped — see below).

Stop-scoped alerts join directly onto `stop_id` at their native grain in
`ml_features` — no resolution step needed. We deliberately do **not** build
a `stop_id -> route_id` fan-out join: one stop alert would spread across
every route serving that stop (mean 10.9 routes/alert among the
route-tagged population, max 1,555), manufacturing false positives and
blurring what should be a precise signal. So this notebook produces two
separate tables, kept on separate keys, and does not merge them into one
flag — a downstream A/B needs to be able to tell which grain (route-tagged
vs. stop-tagged) actually carries predictive signal.

## Point-in-time construction (leakage safety)

Both tables are point-in-time **by construction**: an alert counts for a
given `(key, date, hour)` cell only if it was actually present in a
snapshot captured during that hour — never backfilled into hours before it
was first published. `alert_date` / `alert_hour` are derived from the
snapshot's own capture time (the `YYYY-MM-DD/HH-MM-SS.json` path in the
archive), confirmed below to already be Australia/Brisbane local time
(Brisbane has no DST, so this is unambiguous — verified by converting a raw
GTFS-RT epoch `timestamp` field for a known file and matching it exactly
against that file's path).

In [1]:
# Cell 1 — Environment setup: load .env, configure S3 access, display options
import os
import subprocess
import sys
import re
import json
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv, find_dotenv

try:
    import s3fs
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 's3fs', '-q'], check=True)
    import s3fs

import numpy as np
import pandas as pd

_dotenv_path = find_dotenv(usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path, override=False)
    print(f'Loaded .env from: {_dotenv_path}')
else:
    print('No .env file found — using defaults.')

S3_BUCKET = os.environ.get('AWS_S3_BUCKET', '')
AWS_REGION = os.environ.get('AWS_REGION', 'ap-southeast-2')
REPO_DIR   = os.environ.get('TRANSIT_AI_REPO_DIR', str(Path.cwd().parent))

if not S3_BUCKET:
    raise EnvironmentError('AWS_S3_BUCKET is not set. Check your .env file.')

fs = s3fs.S3FileSystem()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(f'S3_BUCKET: {S3_BUCKET}')
print(f'AWS_REGION: {AWS_REGION}')
print(f'REPO_DIR: {REPO_DIR}')
print(f's3fs: {s3fs.__version__}')

Loaded .env from: /Users/proteeksanyal/Desktop/Learning/Transit-AI/.env
S3_BUCKET: seq-transit-ai-data-ps
AWS_REGION: ap-southeast-2
REPO_DIR: /Users/proteeksanyal/Desktop/Learning/Transit-AI
s3fs: 2025.10.0


## Effect severity ordering

`route_alert_effect` / `stop_alert_effect` report the single **most severe**
effect among the alerts active in a given cell. Ordering below is as
specified, extended with two values that exist in this feed's real data but
weren't in the original spec (`ADDITIONAL_SERVICE`, `NO_EFFECT`) — both
placed at the bottom since neither represents genuine disruption;
`NO_EFFECT` is the least severe of all by definition.

| rank | effect | severity |
|---|---|---|
| 1 (highest) | `NO_SERVICE` | not observed in this feed to date; included defensively per spec |
| 2 | `REDUCED_SERVICE` | |
| 3 | `SIGNIFICANT_DELAYS` | |
| 4 | `DETOUR` | |
| 5 | `STOP_MOVED` | |
| 6 | `MODIFIED_SERVICE` | |
| 7 | `ACCESSIBILITY_ISSUE` | |
| 8 | `OTHER_EFFECT` | |
| 9 | `ADDITIONAL_SERVICE` | extended — a positive addition, not a disruption |
| 10 (lowest) | `NO_EFFECT` | extended — explicitly no impact by definition |

Any effect value not in this table (none observed in practice) sinks below
`NO_EFFECT` defensively rather than raising.

In [2]:
# Cell 2 — Effect severity ranking
SEVERITY_ORDER = [
    'NO_SERVICE', 'REDUCED_SERVICE', 'SIGNIFICANT_DELAYS', 'DETOUR', 'STOP_MOVED',
    'MODIFIED_SERVICE', 'ACCESSIBILITY_ISSUE', 'OTHER_EFFECT',
    'ADDITIONAL_SERVICE', 'NO_EFFECT',
]
EFFECT_SEVERITY_RANK = {name: i for i, name in enumerate(SEVERITY_ORDER)}
UNKNOWN_EFFECT_RANK = len(SEVERITY_ORDER)  # sinks below NO_EFFECT, i.e. least severe
RANK_TO_EFFECT = {v: k for k, v in EFFECT_SEVERITY_RANK.items()}

print('Effect severity ranks (1 = most severe):')
for name, rank in EFFECT_SEVERITY_RANK.items():
    print(f'  {rank + 1:>2}. {name}')

Effect severity ranks (1 = most severe):
   1. NO_SERVICE
   2. REDUCED_SERVICE
   3. SIGNIFICANT_DELAYS
   4. DETOUR
   5. STOP_MOVED
   6. MODIFIED_SERVICE
   7. ACCESSIBILITY_ISSUE
   8. OTHER_EFFECT
   9. ADDITIONAL_SERVICE
  10. NO_EFFECT


## `is_rail_replacement` — derived from confirmed feed wording, not guessed

A targeted re-scan of the `service_alerts` archive (diagnostic 3a follow-up)
printed the full, untruncated text of every alert matching a broad
candidate-term list, so the regex below is built from **confirmed phrases
this feed actually uses**, not assumed ones:

- `"rail replacement"` / `"replacement bus(es)"` — literal, e.g. *"Rail
  replacement buses travelling along the Pacific Motorway... are delayed"*,
  *"Rail replacement buses have been arranged to keep you moving."*
- `"railbus"` — operator-specific term, e.g. *"the railbus services at
  Nerang station will move..."*
- `"track closure"` — the feed's actual term for planned trackwork (not
  "trackwork" or "track work" as one/two words — those were never observed),
  e.g. *"Weekend track closures - Beenleigh and Gold Coast lines"*.
- `"train(s) cancelled"` / `"train(s) suspended"` — live cancellation
  language, e.g. *"5:47am Gympie North to Central train cancelled"*,
  *"Trains suspended - Ipswich and Springfield lines"*.

**Deliberately excluded**: a bare `"replace"` substring was tested and
produces false positives unrelated to rail replacement — *"work will be
undertaken to **replace** the lift"* (an escalator/elevator outage) and
*"delayed... due to required crew **replacement**"* (a staffing issue, not
a bus-for-train substitution). `"not running"` and standalone
`"bus service"` were also tested and either matched zero alerts or matched
unrelated local-bus timetable changes, so neither is included.

This regex matched **29 distinct alerts** in the verification scan — printed
again below against the live data loaded in this run, since the archive
keeps growing.

This flag matters beyond raw counts: on the Gold Coast <-> Brisbane
corridor, a `STOP_MOVED` alert frequently marks a temporary
replacement-bus stop standing in for a cancelled train (confirmed operator
knowledge, e.g. Coomera) — a high-severity service event, not a cosmetic
stop shift. See the stop-level significance note further down.

In [3]:
# Cell 3 — Rail-replacement regex, derived from confirmed archive wording (see markdown above)
RAIL_REPLACEMENT_PATTERN = re.compile(
    r'rail replacement|replacement buses?|railbus|track closure|trains?\s+(?:cancelled|suspended)',
    re.IGNORECASE,
)
print(f'RAIL_REPLACEMENT_PATTERN: {RAIL_REPLACEMENT_PATTERN.pattern}')

RAIL_REPLACEMENT_PATTERN: rail replacement|replacement buses?|railbus|track closure|trains?\s+(?:cancelled|suspended)


## Load the full `service_alerts` archive

Same `YYYY-MM-DD/HH-MM-SS.json` prefix convention as `trip_updates`. ~1,500+
small files, so a bounded thread pool is used purely for I/O parallelism —
no computation changes, matches the approach already proven in the prior
diagnostic passes over this same archive.

In [4]:
# Cell 4 — List all service_alerts snapshot files and parse each path's capture date/hour
ALERT_PREFIX = f'{S3_BUCKET}/gtfs_realtime/service_alerts'
date_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')
time_pattern = re.compile(r'^(\d{2})-(\d{2})-(\d{2})\.json$')

entries = fs.ls(ALERT_PREFIX)
alert_dates = sorted([e.rstrip('/').split('/')[-1] for e in entries if date_pattern.match(e.rstrip('/').split('/')[-1])])

alert_files = []  # (capture_date, capture_hour, path)
for d in alert_dates:
    for fpath in fs.ls(f'{ALERT_PREFIX}/{d}'):
        fname = fpath.split('/')[-1]
        m = time_pattern.match(fname)
        if m:
            alert_files.append((d, int(m.group(1)), fpath))

print(f'service_alerts prefix: s3://{ALERT_PREFIX}/')
print(f'Date folders: {len(alert_dates)}  ({alert_dates[0]} .. {alert_dates[-1]})')
print(f'Snapshot files: {len(alert_files)}')

service_alerts prefix: s3://seq-transit-ai-data-ps/gtfs_realtime/service_alerts/
Date folders: 21  (2026-06-28 .. 2026-07-18)
Snapshot files: 1558


In [5]:
# Cell 5 — Load every snapshot and flatten to one row per (snapshot, alert) occurrence
def load_json(item):
    d, h, fpath = item
    with fs.open(fpath, 'rb') as f:
        return d, h, json.load(f)

records = []
with ThreadPoolExecutor(max_workers=32) as ex:
    futs = [ex.submit(load_json, item) for item in alert_files]
    for i, fut in enumerate(as_completed(futs)):
        d, h, data = fut.result()
        for a in data:
            entities = a.get('informed_entities', [])
            route_ids = sorted({e.get('route_id') for e in entities if e.get('route_id')})
            stop_ids = sorted({e.get('stop_id') for e in entities if e.get('stop_id')})
            records.append({
                'alert_date': d,
                'alert_hour': h,
                'alert_id': a.get('alert_id'),
                'cause': a.get('cause'),
                'effect': a.get('effect'),
                'header': a.get('header') or '',
                'description': a.get('description') or '',
                'route_ids': route_ids,
                'stop_ids': stop_ids,
            })
        if (i + 1) % 400 == 0:
            print(f'  loaded {i + 1}/{len(alert_files)} files...')

alerts_raw = pd.DataFrame(records)
print(f'\nTotal (snapshot, alert) occurrence rows: {len(alerts_raw):,}')

  loaded 400/1558 files...


  loaded 800/1558 files...


  loaded 1200/1558 files...



Total (snapshot, alert) occurrence rows: 105,595


## Drop occurrences with no `informed_entity` at all

These can't be resolved to either grain — dropped and counted explicitly
rather than silently discarded.

In [6]:
# Cell 6 — Drop occurrences with zero informed_entity of any kind
has_route = alerts_raw['route_ids'].apply(len) > 0
has_stop = alerts_raw['stop_ids'].apply(len) > 0
no_entity_mask = ~has_route & ~has_stop

n_dropped = int(no_entity_mask.sum())
n_total_raw = len(alerts_raw)
print(f'Dropped (no informed_entity at all): {n_dropped:,} / {n_total_raw:,} ({n_dropped / n_total_raw * 100:.2f}%)')

alerts_long = alerts_raw.loc[~no_entity_mask].copy()
print(f'Remaining resolvable occurrences: {len(alerts_long):,}')

Dropped (no informed_entity at all): 874 / 105,595 (0.83%)
Remaining resolvable occurrences: 104,721


## Apply the rail-replacement flag to the live data

In [7]:
# Cell 7 — Compute is_rail_replacement per alert occurrence
text = (alerts_long['header'] + ' ' + alerts_long['description']).str.lower()
alerts_long['is_rail_replacement'] = text.str.contains(RAIL_REPLACEMENT_PATTERN, regex=True, na=False)

n_matched_occurrences = int(alerts_long['is_rail_replacement'].sum())
n_matched_alerts = alerts_long.loc[alerts_long['is_rail_replacement'], 'alert_id'].nunique()
print(f'RAIL_REPLACEMENT_PATTERN matched {n_matched_occurrences:,} occurrence rows '
      f'({n_matched_alerts:,} distinct alert_id) in this run\'s live data.')

RAIL_REPLACEMENT_PATTERN matched 9,711 occurrence rows (29 distinct alert_id) in this run's live data.


## Live check: route-resolvability by `cause`

Reproduces the diagnostic 3a finding on the data actually loaded in this
run (no extra S3 reads — reuses `alerts_long` already in memory). Backs the
hard-limitation figures quoted in the documentation section below; numbers
may drift slightly from the original 17.7% / 48.2% as the archive grows,
but should stay in the same range.

In [8]:
# Cell 8 — Live resolvability-by-cause check (reuses in-memory data, no new S3 reads)
cause_check = alerts_long.assign(has_route=alerts_long['route_ids'].apply(len) > 0)
cause_resolve = cause_check.groupby('cause', observed=True)['has_route'].agg(['sum', 'count'])
cause_resolve['pct_resolvable_to_route'] = cause_resolve['sum'] / cause_resolve['count'] * 100
cause_resolve = cause_resolve.rename(columns={'sum': 'n_with_route_id', 'count': 'n_total'})
cause_resolve = cause_resolve.sort_values('n_total', ascending=False)
print(cause_resolve.to_string())

                   n_with_route_id  n_total  pct_resolvable_to_route
cause                                                               
CONSTRUCTION                  8380    47433                17.667025
OTHER_CAUSE                  23698    28234                83.934264
MAINTENANCE                   6974    14509                48.066717
DEMONSTRATION                  721     6940                10.389049
HOLIDAY                       5716     5716               100.000000
STRIKE                        1558     1558               100.000000
ACCIDENT                       243      261                93.103448
POLICE_ACTIVITY                 34       34               100.000000
MEDICAL_EMERGENCY               26       26               100.000000
TECHNICAL_PROBLEM                9        9               100.000000
UNKNOWN_CAUSE                    1        1               100.000000


## Build `route_hourly`

Grain: `(route_id, alert_date, alert_hour)`. Source: occurrences with >=1
`route_id`. An alert naming multiple explicit `route_id`s (mean 10.9, max
1,555 among this population) legitimately contributes to each one — the
producer named them directly. This is not the forbidden stop-to-route
fan-out; it's using exactly the routes the alert itself lists.

In [9]:
# Cell 9 — route_hourly: explode route_ids and aggregate
route_source = alerts_long.loc[has_route.reindex(alerts_long.index, fill_value=False)].copy()
route_exploded = route_source.explode('route_ids').rename(columns={'route_ids': 'route_id'})
route_exploded['effect_rank'] = route_exploded['effect'].map(EFFECT_SEVERITY_RANK).fillna(UNKNOWN_EFFECT_RANK)

grp = route_exploded.groupby(['route_id', 'alert_date', 'alert_hour'], observed=True)
route_hourly = grp.agg(
    route_n_alerts=('alert_id', 'nunique'),
    is_construction_routelevel=('cause', lambda s: bool((s == 'CONSTRUCTION').any())),
    is_maintenance_routelevel=('cause', lambda s: bool((s == 'MAINTENANCE').any())),
    is_rail_replacement_routelevel=('is_rail_replacement', 'any'),
    _min_effect_rank=('effect_rank', 'min'),
).reset_index()

route_hourly['route_alert_effect'] = route_hourly['_min_effect_rank'].map(RANK_TO_EFFECT)
route_hourly['route_has_alert'] = 1
route_hourly = route_hourly.drop(columns='_min_effect_rank')
route_hourly = route_hourly[[
    'route_id', 'alert_date', 'alert_hour', 'route_has_alert', 'route_n_alerts',
    'route_alert_effect', 'is_construction_routelevel', 'is_maintenance_routelevel',
    'is_rail_replacement_routelevel',
]]

print(f'route_hourly rows: {len(route_hourly):,}')
print(route_hourly.dtypes)
route_hourly.head()

route_hourly rows: 266,831
route_id                          object
alert_date                        object
alert_hour                         int64
route_has_alert                    int64
route_n_alerts                     int64
route_alert_effect                object
is_construction_routelevel          bool
is_maintenance_routelevel           bool
is_rail_replacement_routelevel      bool
dtype: object


,route_id,alert_date,alert_hour,route_has_alert,route_n_alerts,route_alert_effect,is_construction_routelevel,is_maintenance_routelevel,is_rail_replacement_routelevel
0,100-4799,2026-06-29,7,1,2,REDUCED_SERVICE,False,False,False
1,100-4799,2026-06-29,8,1,3,REDUCED_SERVICE,False,False,False
2,100-4799,2026-06-29,9,1,2,REDUCED_SERVICE,False,False,False
3,100-4799,2026-07-01,23,1,1,REDUCED_SERVICE,False,False,False
4,100-4799,2026-07-02,6,1,1,REDUCED_SERVICE,False,False,False


## Build `stop_hourly`

Grain: `(stop_id, alert_date, alert_hour)`. Source: occurrences with a
`stop_id` but **no** `route_id` at all — the 54.3% bucket that cannot be
route-resolved. This is the larger of the two tables and is **not**
assumed low-signal (see documentation below).

In [10]:
# Cell 10 — stop_hourly: explode stop_ids (route-less occurrences only) and aggregate
stop_source_mask = (~has_route) & has_stop
stop_source = alerts_long.loc[stop_source_mask.reindex(alerts_long.index, fill_value=False)].copy()
stop_exploded = stop_source.explode('stop_ids').rename(columns={'stop_ids': 'stop_id'})
stop_exploded['effect_rank'] = stop_exploded['effect'].map(EFFECT_SEVERITY_RANK).fillna(UNKNOWN_EFFECT_RANK)

grp = stop_exploded.groupby(['stop_id', 'alert_date', 'alert_hour'], observed=True)
stop_hourly = grp.agg(
    stop_n_alerts=('alert_id', 'nunique'),
    is_construction_stoplevel=('cause', lambda s: bool((s == 'CONSTRUCTION').any())),
    is_maintenance_stoplevel=('cause', lambda s: bool((s == 'MAINTENANCE').any())),
    is_rail_replacement_stoplevel=('is_rail_replacement', 'any'),
    _min_effect_rank=('effect_rank', 'min'),
).reset_index()

stop_hourly['stop_alert_effect'] = stop_hourly['_min_effect_rank'].map(RANK_TO_EFFECT)
stop_hourly['stop_has_alert'] = 1
stop_hourly = stop_hourly.drop(columns='_min_effect_rank')
stop_hourly = stop_hourly[[
    'stop_id', 'alert_date', 'alert_hour', 'stop_has_alert', 'stop_n_alerts',
    'stop_alert_effect', 'is_construction_stoplevel', 'is_maintenance_stoplevel',
    'is_rail_replacement_stoplevel',
]]

print(f'stop_hourly rows: {len(stop_hourly):,}')
print(stop_hourly.dtypes)
stop_hourly.head()

stop_hourly rows: 30,940
stop_id                          object
alert_date                       object
alert_hour                        int64
stop_has_alert                    int64
stop_n_alerts                     int64
stop_alert_effect                object
is_construction_stoplevel          bool
is_maintenance_stoplevel           bool
is_rail_replacement_stoplevel      bool
dtype: object


,stop_id,alert_date,alert_hour,stop_has_alert,stop_n_alerts,stop_alert_effect,is_construction_stoplevel,is_maintenance_stoplevel,is_rail_replacement_stoplevel
0,10150,2026-07-06,14,1,1,STOP_MOVED,True,False,False
1,10150,2026-07-06,15,1,1,STOP_MOVED,True,False,False
2,10150,2026-07-06,16,1,1,STOP_MOVED,True,False,False
3,10150,2026-07-06,17,1,1,STOP_MOVED,True,False,False
4,10150,2026-07-06,18,1,1,STOP_MOVED,True,False,False


## Required documentation

**a) Hard limitation on naming** — `is_construction_routelevel` /
`is_maintenance_routelevel` mean **"construction/maintenance EXPLICITLY
TAGGED to this route,"** not "construction/maintenance affecting this
route." At diagnostic time, only **17.7%** of `CONSTRUCTION`-caused alert
occurrences and **48.2%** of `MAINTENANCE`-caused occurrences carried any
`route_id` at all — the rest were stop-tagged only, and end up in
`stop_hourly` (`is_construction_stoplevel` / `is_maintenance_stoplevel`)
instead. The live equivalent numbers for this run are printed in the "Live
check" cell above. **Do not treat the route-level flags as complete
construction/maintenance coverage for a route** — most of that signal
lives at the stop grain.

**b) Stop-level alerts are not low-signal** — the majority-share grain
(`stop_hourly`, 54.3% of resolvable occurrences) is not a fallback bucket
for noise. On the Gold Coast <-> Brisbane corridor specifically, a
`STOP_MOVED` alert frequently marks a temporary replacement-bus stop
standing in for a cancelled train (confirmed operator knowledge, e.g.
Coomera) — i.e. `is_rail_replacement_stoplevel=True` at a stop can indicate
a *train cancellation*, one of the highest-severity events this dataset can
express, not a cosmetic stop relocation. Consumers should not down-weight
`stop_hourly` relative to `route_hourly` by default.

**c) Survivorship-bias caveat** — if a cancelled/suspended service simply
stops emitting `trip_updates` (or emits them with a null `delay_seconds`),
the worst disruptions can be *absent* from the training label entirely,
meaning alert features would be evaluated against a label that systematically
excludes the very events they're meant to predict. Measured
`delay_seconds`/`delay_minutes` null rates by mode in the current
`ml_features` snapshot (diagnostic 3a, echoed here — not re-derived):

| mode | rows | null % |
|---|---|---|
| bus | 19,023,548 | 2.51% |
| ferry | 226,124 | **15.75%** |
| rail | 1,157,759 | 1.93% |

Nulls did **not** cluster on particular `source_date`s — the daily null
rate stayed flat in the 2.3%-3.5% band across the full 21-day window, and no
(mode, date) cell exceeded 2x that mode's own baseline null rate. So the
elevated-null risk here is a **persistent ferry-feed characteristic**, not
a date-specific incident tied to any alert event — but it's still worth
flagging that ferry-mode delay labels are the least trustworthy of the
three modes, independent of anything in this notebook.

## Output checks: coverage

In [11]:
# Cell 11 — Coverage report for both tables
def report(df, name, key_col):
    print(f'--- {name} ---')
    print(f'  rows: {len(df):,}')
    print(f'  date coverage: {df["alert_date"].min()} .. {df["alert_date"].max()} '
          f'({df["alert_date"].nunique()} distinct dates)')
    print(f'  distinct {key_col} with >=1 alert: {df[key_col].nunique():,}')
    print()

report(route_hourly, 'route_hourly', 'route_id')
report(stop_hourly, 'stop_hourly', 'stop_id')

n_route_occ = len(route_source)
n_stop_occ = len(stop_source)
n_dropped_check = n_dropped

print('=== Occurrence coverage vs. total raw archive ===')
print(f'Total raw occurrences: {n_total_raw:,}')
print(f'  route-source (>=1 route_id):        {n_route_occ:,} ({n_route_occ / n_total_raw * 100:.1f}%)')
print(f'  stop-source (stop_id, no route_id):  {n_stop_occ:,} ({n_stop_occ / n_total_raw * 100:.1f}%)')
print(f'  dropped (no entity at all):          {n_dropped_check:,} ({n_dropped_check / n_total_raw * 100:.2f}%)')
combined_pct = (n_route_occ + n_stop_occ) / n_total_raw * 100
print(f'\nCombined resolvable coverage (route + stop): {combined_pct:.1f}%')

--- route_hourly ---
  rows: 266,831
  date coverage: 2026-06-28 .. 2026-07-18 (21 distinct dates)
  distinct route_id with >=1 alert: 2,076

--- stop_hourly ---
  rows: 30,940
  date coverage: 2026-06-28 .. 2026-07-18 (21 distinct dates)
  distinct stop_id with >=1 alert: 321

=== Occurrence coverage vs. total raw archive ===
Total raw occurrences: 105,595
  route-source (>=1 route_id):        47,360 (44.9%)
  stop-source (stop_id, no route_id):  57,361 (54.3%)
  dropped (no entity at all):          874 (0.83%)

Combined resolvable coverage (route + stop): 99.2%


## Write both tables to S3

Additive-only: new prefix (`alerts/features/`), nothing under `ml_features/`
or the weather table is touched.

In [12]:
# Cell 12 — Write route_hourly.parquet and stop_hourly.parquet to S3
ROUTE_S3_PATH = f's3://{S3_BUCKET}/alerts/features/route_hourly.parquet'
STOP_S3_PATH = f's3://{S3_BUCKET}/alerts/features/stop_hourly.parquet'

with fs.open(ROUTE_S3_PATH, 'wb') as f:
    route_hourly.to_parquet(f, index=False)
print(f'Wrote {len(route_hourly):,} rows to {ROUTE_S3_PATH}')

with fs.open(STOP_S3_PATH, 'wb') as f:
    stop_hourly.to_parquet(f, index=False)
print(f'Wrote {len(stop_hourly):,} rows to {STOP_S3_PATH}')

Wrote 266,831 rows to s3://seq-transit-ai-data-ps/alerts/features/route_hourly.parquet
Wrote 30,940 rows to s3://seq-transit-ai-data-ps/alerts/features/stop_hourly.parquet
